# Social Media Engagement Analyser (Databricks)

Adaptado do kernel Kaggle `social-media-engagement-analyser` para o workspace Databricks do challenge 004.

1. Rode o **LAB 01** (`01_LAB_importando_dados/lab01_carga_csv`) para criar `workspace.social.social_media_dataset`.
2. Widgets: `catalog_name`, `schema_name`, `data_path` (fallback CSV em `dados/` se a tabela Delta ainda nao existir).
3. Colunas = header do CSV (27). **Nao ha** `engagement_rate` nativo — o kernel deriva `engagement / follower_count` (nao views).
4. Comparacao org/ads sem controle e media crua: o dump tem taxa por view quase constante (~0,199). Declare isso.
5. Cluster ou serverless → **Run All**. A celula `%pip` instala `xgboost`, `textblob` e `shap` no job serverless.

In [ ]:
%pip install xgboost textblob shap

In [ ]:
# ============================================================
# 1. Imports & Global Setup
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

from xgboost import XGBClassifier
from textblob import TextBlob

from scipy.sparse import hstack
from tqdm import tqdm
import shap

tqdm.pandas()
np.random.seed(42)

print("Starting Smart-Social Glass-Box pipeline...")


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np


def widget_or_default(name, default):
    try:
        dbutils.widgets.text(name, default)
        value = dbutils.widgets.get(name)
        return value if value else default
    except NameError:
        return default


def normalize_frame(frame):
    if "is_sponsored" in frame.columns:
        frame["is_sponsored"] = frame["is_sponsored"].map(
            lambda x: None if pd.isna(x) else str(x).strip().lower() in ("true", "1", "t")
        )
    if "post_date" in frame.columns:
        frame["post_date"] = pd.to_datetime(frame["post_date"], errors="coerce")
    return frame


def resolve_data_path(explicit):
    marker = "social_media_dataset.csv"
    candidates = []
    if explicit:
        candidates.append(Path(explicit))
    try:
        notebook_path = (
            dbutils.notebook.entry_point.getDbutils()
            .notebook()
            .getContext()
            .notebookPath()
            .get()
        )
        parts = [part for part in notebook_path.split("/") if part]
        if "notebook-marketing-004-social" in parts:
            idx = parts.index("notebook-marketing-004-social")
            repo = "/".join(parts[: idx + 1])
            candidates.append(Path("/Workspace") / repo / "dados")
    except Exception:
        pass
    here = Path.cwd()
    candidates.extend([here / "dados", here.parent / "dados", here.parent.parent / "dados"])
    for candidate in candidates:
        if (candidate / marker).exists():
            return str(candidate)
    raise FileNotFoundError("Rode o LAB 01 ou defina data_path para a pasta dados/.")


catalog_name = widget_or_default("catalog_name", "workspace")
schema_name = widget_or_default("schema_name", "social")
data_path_widget = widget_or_default("data_path", "")

try:
    df = spark.table(f"{catalog_name}.{schema_name}.social_media_dataset").toPandas()
    print(f"loaded Delta {catalog_name}.{schema_name}.social_media_dataset")
except Exception as exc:
    print("Delta indisponivel, lendo CSV:", exc)
    data_path = resolve_data_path(data_path_widget)
    df = pd.read_csv(f"{data_path}/social_media_dataset.csv")
    print(f"loaded CSV from {data_path}")

df = normalize_frame(df)
print("Dataset loaded with shape:", df.shape)
print("Columns:", df.columns.tolist())


In [ ]:
# ============================================================
# 3. Target Construction (Normalized Engagement → 5 Classes)
# ============================================================

print("Constructing engagement target...")

df["likes"] = df["likes"].fillna(0)
df["shares"] = df["shares"].fillna(0)
df["comments_count"] = df["comments_count"].fillna(0)
df["follower_count"] = df["follower_count"].clip(lower=1)

df["engagement"] = (
    df["likes"] +
    2 * df["shares"] +
    0.5 * df["comments_count"]
)

df["engagement_rate"] = df["engagement"] / df["follower_count"]

df["target"] = pd.qcut(
    df["engagement_rate"],
    q=5,
    labels=[0, 1, 2, 3, 4]
).astype(int)

print("Target distribution:")
print(df["target"].value_counts(normalize=True))


In [ ]:
# ============================================================
# 4. Text Construction
# ============================================================

print("Building text field...")

df["text"] = (
    df["content_description"].fillna("") + " " +
    df["hashtags"].fillna("").str.replace(",", " ") + " " +
    df["comments_text"].fillna("")
)


In [ ]:
# ============================================================
# 5. Metadata Feature Engineering
# ============================================================

print("Engineering metadata features...")

df["post_date"] = pd.to_datetime(df["post_date"], errors="coerce")

df["hour"] = df["post_date"].dt.hour.fillna(12)
df["dayofweek"] = df["post_date"].dt.dayofweek.fillna(3)
df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

df["text_length"] = df["text"].str.len()
df["word_count"] = df["text"].str.split().apply(len)
df["hashtag_count"] = df["hashtags"].fillna("").str.count(",")

df["sentiment_polarity"] = df["text"].progress_apply(
    lambda x: TextBlob(x).sentiment.polarity
)
df["sentiment_subjectivity"] = df["text"].progress_apply(
    lambda x: TextBlob(x).sentiment.subjectivity
)

df["log_followers"] = np.log1p(df["follower_count"])
df["is_big_creator"] = (
    df["follower_count"] > df["follower_count"].median()
).astype(int)

df["is_sponsored"] = df["is_sponsored"].astype(int)
df["has_disclosure"] = (df["disclosure_type"] != "none").astype(int)


In [ ]:
# ============================================================
# 6. Encode Categorical Features
# ============================================================

print("Encoding categorical metadata...")

categorical_cols = [
    "platform",
    "content_type",
    "content_category",
    "language",
    "audience_age_distribution",
    "audience_gender_distribution",
    "audience_location"
]

df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)


In [ ]:
# ============================================================
# 7. Feature List
# ============================================================

base_features = [
    "hour", "dayofweek", "is_weekend",
    "text_length", "word_count", "hashtag_count",
    "log_followers", "is_big_creator",
    "is_sponsored", "has_disclosure",
    "sentiment_polarity", "sentiment_subjectivity"
]

one_hot_features = [
    c for c in df.columns
    if c.startswith((
        "platform_", "content_type_", "content_category_",
        "language_", "audience_age_distribution_",
        "audience_gender_distribution_", "audience_location_"
    ))
]

features = base_features + one_hot_features
print("Total metadata features:", len(features))


In [ ]:
# ============================================================
# 8. Group-Aware Train / Val / Test Split
# ============================================================

print("Performing creator-level split...")

gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss.split(df, groups=df["creator_id"]))

df_train = df.iloc[train_idx]
df_temp  = df.iloc[temp_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(gss2.split(df_temp, groups=df_temp["creator_id"]))

df_val  = df_temp.iloc[val_idx]
df_test = df_temp.iloc[test_idx]

print("Train size:", df_train.shape)
print("Val size:", df_val.shape)
print("Test size:", df_test.shape)


In [ ]:
# ============================================================
# 9. TF-IDF Vectorization (Train Only)
# ============================================================

print("Fitting TF-IDF vectorizer...")

tfidf = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    min_df=5,
    stop_words="english"
)

X_text_train = tfidf.fit_transform(df_train["text"])
X_text_val   = tfidf.transform(df_val["text"])
X_text_test  = tfidf.transform(df_test["text"])


In [ ]:
# ============================================================
# 10. Fusion Feature Matrix
# ============================================================

print("Building fusion feature matrices...")

X_meta_train = df_train[features].astype(np.float32)
X_meta_val   = df_val[features].astype(np.float32)
X_meta_test  = df_test[features].astype(np.float32)

X_train_all = hstack([X_text_train, X_meta_train.values]).tocsr()
X_val_all   = hstack([X_text_val,   X_meta_val.values]).tocsr()
X_test_all  = hstack([X_text_test,  X_meta_test.values]).tocsr()

y_train = df_train["target"]
y_val   = df_val["target"]
y_test  = df_test["target"]


In [ ]:
# ============================================================
# 11. Train Fusion XGBoost Model
# ============================================================

print("Training fusion XGBoost classifier...")

fusion_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="multi:softprob",
    num_class=5,
    eval_metric="mlogloss",
    random_state=42
)

fusion_xgb.fit(X_train_all, y_train)

val_preds  = fusion_xgb.predict(X_val_all)
test_preds = fusion_xgb.predict(X_test_all)

print("Validation Accuracy:", accuracy_score(y_val, val_preds))
print("Test Accuracy:", accuracy_score(y_test, test_preds))
print(classification_report(y_test, test_preds))


In [ ]:
cm = confusion_matrix(y_test, test_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
y_test_bin = pd.get_dummies(y_test)
y_prob = fusion_xgb.predict_proba(X_test_all)

roc_auc = roc_auc_score(y_test_bin, y_prob, multi_class="ovr")
print("Macro ROC-AUC:", roc_auc)

plt.figure(figsize=(6, 5))
for i in range(5):
    fpr, tpr, _ = roc_curve(y_test_bin.iloc[:, i], y_prob[:, i])
    plt.plot(fpr, tpr, label=f"Class {i}")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()
plt.show()


In [ ]:
feature_names = np.concatenate([
    tfidf.get_feature_names_out(),
    X_meta_train.columns
])

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": fusion_xgb.feature_importances_
}).sort_values(by="importance", ascending=False)

print("Top 20 important features:")
print(importance_df.head(20))


In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(
    df[base_features + ["target"]].corr(),
    cmap="coolwarm",
    center=0
)
plt.title("Metadata Correlation Matrix")
plt.show()


In [ ]:
df.groupby("hour")["target"].mean().plot(figsize=(7, 4))
plt.title("Average Engagement by Hour")
plt.ylabel("Engagement Class")
plt.show()


In [ ]:
df.groupby("is_sponsored")["target"].mean().plot(
    kind="bar",
    figsize=(5, 4)
)
plt.title("Engagement vs Sponsorship")
plt.xticks([0, 1], ["Organic", "Sponsored"], rotation=0)
plt.show()


In [ ]:
print("Initializing SHAP explainer with background data...")

X_shap = X_test_all[:200].toarray()
explainer = shap.TreeExplainer(
    fusion_xgb,
    feature_names=feature_names,
    feature_perturbation="tree_path_dependent",
)
shap_values = explainer.shap_values(X_shap)
if isinstance(shap_values, list):
    class_shap = shap_values[4]
elif getattr(shap_values, "ndim", 0) == 3:
    class_shap = shap_values[:, :, 4]
else:
    class_shap = shap_values

shap.summary_plot(
    class_shap,
    X_shap,
    feature_names=feature_names,
    max_display=20,
)


In [ ]:
df["creator_bucket"] = pd.qcut(
    df["follower_count"],
    q=3,
    labels=["Small", "Medium", "Large"]
)

plt.figure(figsize=(8, 4))
sns.barplot(
    data=df,
    x="creator_bucket",
    y="target",
    hue="is_sponsored",
    ci=95
)

plt.title("Engagement vs Sponsorship (Controlled by Creator Size)")
plt.ylabel("Average Engagement Class")
plt.xlabel("Creator Size")
plt.show()


In [ ]:
# ============================================================
# Save trained model and preprocessing artifacts locally
# ============================================================

import os
import joblib

# Directory to store artifacts
ARTIFACT_DIR = "artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# Paths
model_path = os.path.join(ARTIFACT_DIR, "fusion_xgb_model.joblib")
tfidf_path = os.path.join(ARTIFACT_DIR, "tfidf_vectorizer.joblib")
features_path = os.path.join(ARTIFACT_DIR, "metadata_features.joblib")

# Save artifacts
joblib.dump(fusion_xgb, model_path)
joblib.dump(tfidf, tfidf_path)
joblib.dump(features, features_path)

print("Artifacts saved successfully:")
print("Model:", model_path)
print("TF-IDF Vectorizer:", tfidf_path)
print("Metadata Features:", features_path)
